# 01.8 - Tries (Prefix Trees)

## What is a trie?
A tree where each **edge is a character** and each path from the root spells a prefix. Words sharing a prefix share nodes.

```
words: car, cat, cart, dog

        (root)
        /    \
       c      d
       |      |
       a      o
      / \     |
     r*  t*   g*        * = end of a word
     |
     t*
```
Insert / search / prefix-check cost **O(L)** (L = word length) - independent of how many words are stored!

## Where is it used?
| Real system | Use |
|---|---|
| **Search-box autocomplete** (Google, YouTube, IDEs, phone keyboards) | All words starting with what you typed |
| **Spell checkers** | Fast dictionary lookups + suggestions |
| **Internet routers** | **Longest-prefix match** of IP addresses |
| **Phone contacts / T9** | Prefix search |
| **Word games** (Boggle, Scrabble solvers) | Prune paths that aren't prefixes |
| **Search engines / text indexing** | Prefix & suffix trees |

## 1. Build a trie - also LC #208 Implement Trie (Medium) 

In [1]:
class TrieNode:
    def __init__(self):
        self.children = {}          # char -> TrieNode
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):         # O(L)
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):         # O(L)
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix):   # O(L)
        return self._walk(prefix) is not None

    def autocomplete(self, prefix, limit=10):
        node = self._walk(prefix)
        results = []
        def dfs(n, path):
            if len(results) >= limit:
                return
            if n.is_end:
                results.append(path)
            for ch in sorted(n.children):
                dfs(n.children[ch], path + ch)
        if node:
            dfs(node, prefix)
        return results

t = Trie()
for w in ["apple", "app", "application", "apply", "banana", "band", "bandwidth"]:
    t.insert(w)
assert t.search("app") and not t.search("appl") and t.startsWith("appl")
print("autocomplete('app'):", t.autocomplete("app"))
print("autocomplete('ban'):", t.autocomplete("ban"))
print("Trie ")

autocomplete('app'): ['app', 'apple', 'application', 'apply']
autocomplete('ban'): ['banana', 'band', 'bandwidth']
Trie 


---
## Solved problems

### 1. Design Add and Search Words - LC #211 (Medium)
`search` supports `.` matching any single letter.

**Idea:** Normal trie; on `.`, try every child (DFS).

In [2]:
class WordDictionary:
    def __init__(self):
        self.root = TrieNode()

    def addWord(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search(self, word):
        def dfs(node, i):
            if i == len(word):
                return node.is_end
            ch = word[i]
            if ch == ".":
                return any(dfs(child, i + 1) for child in node.children.values())
            return ch in node.children and dfs(node.children[ch], i + 1)
        return dfs(self.root, 0)

wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.addWord(w)
assert not wd.search("pad") and wd.search("bad") and wd.search(".ad") and wd.search("b..")
print("Add and Search Words ")

Add and Search Words 


### 2. Replace Words - LC #648 (Medium)
Replace each word in a sentence with the shortest dictionary root that is its prefix.

In [3]:
def replace_words(dictionary, sentence):
    trie = Trie()
    for root in dictionary:
        trie.insert(root)

    def shortest_root(word):
        node = trie.root
        for i, ch in enumerate(word):
            if ch not in node.children:
                break
            node = node.children[ch]
            if node.is_end:
                return word[:i + 1]
        return word

    return " ".join(shortest_root(w) for w in sentence.split())

assert replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
assert replace_words(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"
print("Replace Words ")

Replace Words 


### 3. Word Search II - LC #212 (Hard) Find all dictionary words that can be traced in a letter grid (adjacent cells, no reuse). The Boggle problem.

**Idea:** Put all words in a trie, then DFS from every cell walking the trie at the same time. Stop as soon as the path isn't a prefix of any word.

In [4]:
def find_words(board, words):
    trie = Trie()
    for w in words:
        trie.insert(w)
    rows, cols = len(board), len(board[0])
    found = set()

    def dfs(r, c, node, path):
        ch = board[r][c]
        if ch not in node.children:
            return                              # prune: no word has this prefix
        node = node.children[ch]
        path += ch
        if node.is_end:
            found.add(path)
        board[r][c] = "#"                       # mark used
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] != "#":
                dfs(nr, nc, node, path)
        board[r][c] = ch                        # restore (backtrack)

    for r in range(rows):
        for c in range(cols):
            dfs(r, c, trie.root, "")
    return sorted(found)

board = [["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]]
assert find_words(board, ["oath", "pea", "eat", "rain"]) == ["eat", "oath"]
print("Word Search II ")

Word Search II 


## More practice on LeetCode
Medium: Longest Word in Dictionary (#720), Map Sum Pairs (#677), Search Suggestions System (#1268)
Hard: Palindrome Pairs (#336)

## Key takeaways
- Trie = tree of characters; O(L) insert/search regardless of dictionary size.
- Perfect for **prefix** questions: autocomplete, "starts with", longest prefix.
- Node = `dict` of children + `is_end` flag.

**Next: `01.9-Union-Find.ipynb`**